In [17]:
import numpy as np

def solve_qp_ipm(P, q, A, b, tol=1e-14, max_iter=10000):
    """
    Solve:
        min 0.5 x^T P x + q^T x
        s.t. A x = b
             x >= 0
    """

    n = P.shape[0]
    m = A.shape[0]

    # Initial point (strictly feasible)
    x = np.ones(n)
    s = np.ones(n)
    y = np.zeros(m)

    e = np.ones(n)

    for it in range(max_iter):
        # Residuals
        r_dual = P @ x + q - A.T @ y - s
        r_pri  = A @ x - b
        mu = (x @ s) / n
        r_cent = x * s - mu * e

        # Check convergence
        if (np.linalg.norm(r_dual) < tol and
            np.linalg.norm(r_pri) < tol and
            mu < tol):
            break

        # KKT matrix blocks
        X = np.diag(x)
        S = np.diag(s)

        # Build KKT system
        KKT = np.block([
            [P,         -A.T,      -np.eye(n)],
            [A,         np.zeros((m, m)), np.zeros((m, n))],
            [S,         np.zeros((n, m)), X]
        ])

        rhs = -np.concatenate([r_dual, r_pri, r_cent])

        # Solve Newton system
        delta = np.linalg.solve(KKT, rhs)

        dx = delta[:n]
        dy = delta[n:n+m]
        ds = delta[n+m:]

        # Step size (maintain positivity)
        alpha = 1.0
        idx = dx < 0
        if np.any(idx):
            alpha = min(alpha, 0.99 * np.min(-x[idx] / dx[idx]))

        idx = ds < 0
        if np.any(idx):
            alpha = min(alpha, 0.99 * np.min(-s[idx] / ds[idx]))

        # Update
        x += alpha * dx
        y += alpha * dy
        s += alpha * ds

    return x, y, s

P = np.array([[4, 1],
              [1, 2]])
q = np.array([1, 1])

A = np.array([[1, 1]])
b = np.array([1])

x, y, s = solve_qp_ipm(P, q, A, b)
x_t = np.array([0, 1])

print("Solution x:", x)
print(0.5 * x.T @ P @ x + q.T @ x)
print(0.5 * x_t.T @ P @ x_t + q.T @ x_t)

Solution x: [0.40090598 0.59909402]
1.92054522760254
2.0
